# SegNet Inference Demo

Kurzer Testlauf für das Segmentierungsnetzwerk: wir ziehen zufällig 5 Bilder aus einem Eingabe-Ordner, laden automatisch den neuesten `segnet_full.pth` aus `experiments/seg_*` und speichern farbcodierte Masken (auf 16:9 gestreckt) plus Side-by-Side-Visualisierungen samt Legende unter `results/`.

In [ ]:
from pathlib import Path

import os

os.chdir("/srv/store/docker-users/thesis/khajuria/day2night")
print("Current working directory:", os.getcwd())


# Pfade anpassen, falls nötig
#INPUT_DIR = "data/Straßen Bilder/test"  # Quelle der Testbilder
INPUT_DIR = "data/fh_nacht/nacht/train"
OUTPUT_DIR = "results/segnet_notebook"        # Basis-Ordner für Ergebnisse
CONFIG = "configs/seg.yaml"                    # Für Transform-Settings (Resize/CenterCrop)
CHECKPOINT = "experiments/seg_20251204_174637/segnet_full.pth"                               # Optional: Pfad zu segnet_full.pth, sonst neuester seg_* Run
NUM_SAMPLES = 5
RUN_NAME = Path(CHECKPOINT).parents[0].name if CHECKPOINT else None
print("Using run name:", RUN_NAME)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    "-m",
    "src.apply_segnet",
    "--input-dir",
    INPUT_DIR,
    "--output-dir",
    OUTPUT_DIR,
    "--config",
    CONFIG,
    "--num-samples",
    str(NUM_SAMPLES),
]
if RUN_NAME:
    cmd += ["--run-name", RUN_NAME]
if CHECKPOINT:
    cmd += ["--checkpoint", CHECKPOINT]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Run-Ordner bestimmen (entweder explizit oder jüngster Unterordner nach dem Lauf)
base = Path(OUTPUT_DIR)
if RUN_NAME:
    run_dir = base / RUN_NAME
else:
    run_dirs = sorted([p for p in base.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise FileNotFoundError(f"Kein Unterordner unter {base} gefunden.")
    run_dir = run_dirs[-1]
print("Run directory:", run_dir)


In [6]:
from IPython.display import display
from PIL import Image

legend_path = run_dir / "legend.png"
if legend_path.is_file():
    print("Legende:")
    display(Image.open(legend_path))

viz_dir = run_dir / "viz"
viz_paths = sorted(viz_dir.glob("*.png"))
if not viz_paths:
    raise FileNotFoundError(f"Keine Visualisierungen unter {viz_dir} gefunden.")

for path in viz_paths:
    print(path.name)
    display(Image.open(path))
